# Phase 7 — Clean Cross-Model Replication

**Role in the project:** a compact, reproducible bridge from Phase 6 to Phase 8.

Model:

\[
\texttt{EleutherAI/pythia-160m}
\]

Primary question:

> At equal patch budget \(k=2\), does a pair selected by measured pair recovery outperform the two best first-order AIE sites on a different Transformer?

The notebook deliberately separates three concepts:

\[
\text{AIE}
\quad\neq\quad
\text{pair recovery}
\quad\neq\quad
\text{synergy }J.
\]

- **AIE pair:** top two single sites by mean normalized indirect effect.
- **Recovery-optimal pair:** pair with maximum mean measured causal recovery.
- **Synergy-optimal pair:** pair with maximum mean \(J\); reported descriptively, not substituted for the primary recovery comparison.

## Design

- Discovery: 32 new CounterFact subjects, disjoint from Phases 5/6.
- Holdout: 128 further subject-disjoint facts.
- Patch budget: \(k=2\).
- Primary noise scale: \(3.0\).
- Holdout noise scales: \(2.5,3.0,3.5\).
- Holdout noise seeds: \(101,202,303\).
- Primary endpoint:

\[
\Delta
=
R_{\mathrm{RECOVERY}}
-
R_{\mathrm{AIE}}.
\]

- Statistical success criterion:

\[
\text{paired bootstrap 95\% CI lower bound}>0.
\]

No manual replacement cells are used. Discovery cases are validated before the manifest is frozen.

## Why this rewrite exists

The previous notebook could accumulate stale discovery rows and replacement cases across repeated Jupyter execution. That produced:

- more than 32 discovery case IDs;
- duplicated keys;
- incompatible values associated with the same key;
- a manual hard-coded pair after discovery.

This version is designed to be **idempotent**:

1. the final 32-case discovery manifest is built once from deterministic rules;
2. the manifest receives a SHA-256 hash;
3. result filenames include a `run_id` derived from that hash and the protocol version;
4. every result row carries the same `run_id`;
5. exact expected row counts and unique-key checks are asserted;
6. selected pairs are computed from the frozen discovery results, never hard-coded.

In [1]:
import os
import json
import hashlib
import platform
import numpy as np
import pandas as pd
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
)
from datasets import load_dataset

PHASE7_VERSION = "phase7_clean_v1"

print("python:", platform.python_version())
print("torch:", torch.__version__)

C:\Users\Mauro\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


python: 3.11.6
torch: 2.4.0+cpu


In [2]:
MODEL_ID = "EleutherAI/pythia-160m"

DEVICE = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("device:", DEVICE)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    use_fast=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
).to(DEVICE).eval()

N_LAYERS = len(model.gpt_neox.layers)

EMBED_STD = (
    model.gpt_neox.embed_in
    .weight
    .detach()
    .std()
    .item()
)

print(type(model))
print("layers:", N_LAYERS)
print("embedding std:", EMBED_STD)

device: cpu


C:\Users\Mauro\AppData\Local\Programs\Python\Python311\Lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


<class 'transformers.models.gpt_neox.modeling_gpt_neox.GPTNeoXForCausalLM'>
layers: 12
embedding std: 0.02854427695274353


## Required Phase-5/6 inputs

These files define subjects that must not re-enter Phase 7:

- `phase5_seen_facts.csv`
- `phase6_fresh_manifest.csv`
- `phase6b_locked_manifest.csv`

The notebook fails explicitly if one is missing rather than silently changing the subject-disjoint protocol.

In [3]:
REQUIRED_INPUTS = [
    "phase5_seen_facts.csv",
    "phase6_fresh_manifest.csv",
    "phase6b_locked_manifest.csv",
]

missing = [
    p for p in REQUIRED_INPUTS
    if not os.path.exists(p)
]

if missing:
    raise FileNotFoundError(
        "Missing prerequisite files: "
        + ", ".join(missing)
    )

phase5_seen = pd.read_csv(
    "phase5_seen_facts.csv"
)
phase6a_seen = pd.read_csv(
    "phase6_fresh_manifest.csv"
)
phase6b_seen = pd.read_csv(
    "phase6b_locked_manifest.csv"
)

def norm_subject(x):
    return str(x).strip().lower()

SUBJ5 = set(
    phase5_seen["subject"].map(norm_subject)
)
SUBJ6A = set(
    phase6a_seen["subject"].map(norm_subject)
)
SUBJ6B = set(
    phase6b_seen["subject"].map(norm_subject)
)

PREVIOUS_SUBJECTS = (
    SUBJ5 | SUBJ6A | SUBJ6B
)

print("Phase 5:", len(SUBJ5))
print("Phase 6A:", len(SUBJ6A))
print("Phase 6B:", len(SUBJ6B))
print(
    "Total previous unique subjects:",
    len(PREVIOUS_SUBJECTS),
)

Phase 5: 64
Phase 6A: 128
Phase 6B: 128
Total previous unique subjects: 314


In [4]:
counterfact = load_dataset(
    "azhx/counterfact",
    split="test",
)

print(counterfact)

Dataset({
    features: ['case_id', 'pararel_idx', 'requested_rewrite', 'paraphrase_prompts', 'neighborhood_prompts', 'attribute_prompts', 'generation_prompts'],
    num_rows: 2191
})


# 1. Frozen protocol constants

In [5]:
N_DISCOVERY = 32
N_HOLDOUT = 128

DISCOVERY_SEED = 20260911
HOLDOUT_SEED = 20260912

REFERENCE_SEED = 424242
REFERENCE_SCALE = 3.0

DISCOVERY_NOISE_SEED = 707
DISCOVERY_NOISE_SCALE = 3.0

PRIMARY_SCALE = 3.0

NOISE_SCALES = [
    2.5,
    3.0,
    3.5,
]

NOISE_SEEDS = [
    101,
    202,
    303,
]

MIN_CLEAN_MARGIN = 1.0
MIN_REFERENCE_TE = 1.0
MIN_DISCOVERY_TE = 0.0
MIN_EVAL_TE = 0.5

PRACTICAL_THRESHOLD = 0.02

print("N discovery:", N_DISCOVERY)
print("N holdout:", N_HOLDOUT)

N discovery: 32
N holdout: 128


# 2. Case preparation

For a CounterFact record we require:

1. true and alternative targets each tokenize to exactly one token;
2. the subject span is identifiable;
3. clean true-vs-alternative margin is at least `MIN_CLEAN_MARGIN`;
4. a **separate reference corruption** produces total effect

\[
TE_{\mathrm{ref}}
=
D_{\mathrm{clean}}
-
D_{\mathrm{ref}}
\ge
\texttt{MIN_REFERENCE_TE}.
\]

The reference seed is not the discovery scan seed.

In [6]:
def one_token_id(text):
    ids = tokenizer.encode(
        " " + text.strip(),
        add_special_tokens=False,
    )

    if len(ids) != 1:
        return None

    return int(ids[0])


def subject_token_positions(
    prompt,
    subject,
):
    enc = tokenizer(
        prompt,
        return_offsets_mapping=True,
        add_special_tokens=False,
    )

    start = prompt.find(subject)

    if start < 0:
        return []

    end = start + len(subject)

    positions = []

    for i, (a, b) in enumerate(
        enc["offset_mapping"]
    ):
        if b > start and a < end:
            positions.append(i)

    return positions


def logit_margin(
    logits,
    true_id,
    alt_id,
):
    return float(
        (
            logits[0, -1, true_id]
            -
            logits[0, -1, alt_id]
        ).item()
    )


def corrupted_embeddings(
    input_ids,
    subj_positions,
    seed,
    noise_multiplier,
):
    embeds = (
        model.gpt_neox.embed_in(
            input_ids
        )
        .detach()
        .clone()
    )

    g = torch.Generator(
        device=embeds.device
    )
    g.manual_seed(int(seed))

    noise = torch.randn(
        embeds.shape,
        generator=g,
        device=embeds.device,
        dtype=embeds.dtype,
    )

    noise *= (
        noise_multiplier
        * EMBED_STD
    )

    for pos in subj_positions:
        embeds[:, pos, :] += (
            noise[:, pos, :]
        )

    return embeds

In [7]:
def prepare_case(
    record,
    dataset_index,
):
    rr = record[
        "requested_rewrite"
    ]

    source_case_id = int(
        record["case_id"]
    )

    subject = rr["subject"]
    prompt = rr["prompt"].format(subject)

    true_text = rr[
        "target_true"
    ]["str"]

    new_text = rr[
        "target_new"
    ]["str"]

    true_id = one_token_id(
        true_text
    )
    new_id = one_token_id(
        new_text
    )

    if (
        true_id is None
        or new_id is None
        or true_id == new_id
    ):
        return None

    subj_positions = (
        subject_token_positions(
            prompt,
            subject,
        )
    )

    if not subj_positions:
        return None

    enc = tokenizer(
        prompt,
        return_tensors="pt",
        add_special_tokens=False,
    )

    input_ids = (
        enc["input_ids"]
        .to(DEVICE)
    )
    attention_mask = (
        enc["attention_mask"]
        .to(DEVICE)
    )

    with torch.inference_mode():
        clean_out = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
        )

    D_clean = logit_margin(
        clean_out.logits,
        true_id,
        new_id,
    )

    if D_clean < MIN_CLEAN_MARGIN:
        return None

    ref_emb = corrupted_embeddings(
        input_ids,
        subj_positions,
        REFERENCE_SEED
        + int(dataset_index),
        REFERENCE_SCALE,
    )

    with torch.inference_mode():
        ref_out = model(
            inputs_embeds=ref_emb,
            attention_mask=attention_mask,
        )

    D_ref = logit_margin(
        ref_out.logits,
        true_id,
        new_id,
    )

    TE_reference = (
        D_clean - D_ref
    )

    if TE_reference < MIN_REFERENCE_TE:
        return None

    return {
        # Assigned only after the manifest is frozen:
        "case_id": None,

        # Immutable source identifiers:
        "source_case_id": source_case_id,
        "dataset_index": int(dataset_index),

        "subject": subject,
        "prompt": prompt,
        "true_text": true_text,
        "new_text": new_text,
        "true_id": int(true_id),
        "new_id": int(new_id),

        "input_ids": input_ids,
        "attention_mask": attention_mask,

        "subject_positions": subj_positions,
        "subject_last": int(
            subj_positions[-1]
        ),
        "query_pos": int(
            input_ids.shape[1] - 1
        ),

        "D_clean_reference":
            float(D_clean),

        "TE_reference":
            float(TE_reference),
    }

# 3. Discovery validity is checked *before* freezing the 32 cases

The old notebook selected 32 cases first and later replaced discovery cases for which the actual discovery corruption had \(TE\le0\). Re-running those replacement cells could accumulate cases.

Here the same discovery corruption criterion is evaluated **during candidate selection**.

For candidate \(c\),

\[
TE_{\mathrm{disc}}
=
D_{\mathrm{clean}}
-
D_0.
\]

Only candidates with

\[
TE_{\mathrm{disc}}>0
\]

enter the final 32-case manifest.

Thus there is no replacement phase.

In [8]:
def discovery_runtime_state(
    case,
    need_hidden_states=False,
):
    with torch.inference_mode():
        clean_out = model(
            input_ids=
                case["input_ids"],

            attention_mask=
                case["attention_mask"],

            output_hidden_states=
                need_hidden_states,
        )

    D_clean = logit_margin(
        clean_out.logits,
        case["true_id"],
        case["new_id"],
    )

    actual_seed = (
        DISCOVERY_NOISE_SEED
        +
        int(case["source_case_id"])
    )

    corrupt_emb = (
        corrupted_embeddings(
            case["input_ids"],
            case["subject_positions"],
            actual_seed,
            DISCOVERY_NOISE_SCALE,
        )
    )

    with torch.inference_mode():
        corrupt_out = model(
            inputs_embeds=
                corrupt_emb,

            attention_mask=
                case["attention_mask"],
        )

    D0 = logit_margin(
        corrupt_out.logits,
        case["true_id"],
        case["new_id"],
    )

    TE = D_clean - D0

    clean_states = (
        clean_out.hidden_states
        if need_hidden_states
        else None
    )

    return (
        clean_states,
        corrupt_emb,
        float(D_clean),
        float(D0),
        float(TE),
    )

In [9]:
rng = np.random.default_rng(
    DISCOVERY_SEED
)

indices = rng.permutation(
    len(counterfact)
)

discovery_cases = []
discovery_subjects = set()

for idx in indices:
    record = counterfact[
        int(idx)
    ]

    subject_key = norm_subject(
        record[
            "requested_rewrite"
        ]["subject"]
    )

    if subject_key in PREVIOUS_SUBJECTS:
        continue

    if subject_key in discovery_subjects:
        continue

    case = prepare_case(
        record,
        int(idx),
    )

    if case is None:
        continue

    (
        _,
        _,
        D_clean,
        D0,
        TE_disc,
    ) = discovery_runtime_state(
        case,
        need_hidden_states=False,
    )

    if TE_disc <= MIN_DISCOVERY_TE:
        continue

    case["D_clean_discovery"] = (
        float(D_clean)
    )
    case["D0_discovery"] = (
        float(D0)
    )
    case["TE_discovery"] = (
        float(TE_disc)
    )

    discovery_cases.append(case)
    discovery_subjects.add(
        subject_key
    )

    if len(discovery_cases) >= N_DISCOVERY:
        break


if len(discovery_cases) != N_DISCOVERY:
    raise RuntimeError(
        f"Only found {len(discovery_cases)} "
        f"valid discovery cases; "
        f"expected {N_DISCOVERY}."
    )

# Sequential experiment IDs are assigned only now.
for case_id, case in enumerate(
    discovery_cases
):
    case["case_id"] = int(case_id)

assert [
    c["case_id"]
    for c in discovery_cases
] == list(range(N_DISCOVERY))

assert len({
    norm_subject(c["subject"])
    for c in discovery_cases
}) == N_DISCOVERY

assert (
    discovery_subjects
    & PREVIOUS_SUBJECTS
) == set()

print(
    "Discovery cases:",
    len(discovery_cases),
)

print(
    "Subject overlap with previous phases:",
    len(
        discovery_subjects
        & PREVIOUS_SUBJECTS
    ),
)

Discovery cases: 32
Subject overlap with previous phases: 0


# 4. Freeze and hash the discovery manifest

The hash becomes part of every discovery result filename.

A different manifest automatically produces a different `run_id`.

In [10]:
DISCOVERY_MANIFEST_PATH = (
    "phase7_discovery_manifest_clean.csv"
)

discovery_manifest = pd.DataFrame([
    {
        "case_id":
            c["case_id"],

        "source_case_id":
            c["source_case_id"],

        "dataset_index":
            c["dataset_index"],

        "subject":
            c["subject"],

        "prompt":
            c["prompt"],

        "true_text":
            c["true_text"],

        "new_text":
            c["new_text"],

        "D_clean_reference":
            c["D_clean_reference"],

        "TE_reference":
            c["TE_reference"],

        "D_clean_discovery":
            c["D_clean_discovery"],

        "D0_discovery":
            c["D0_discovery"],

        "TE_discovery":
            c["TE_discovery"],
    }
    for c in discovery_cases
])

assert len(discovery_manifest) == N_DISCOVERY
assert discovery_manifest["case_id"].is_unique
assert discovery_manifest["subject"].map(
    norm_subject
).is_unique

discovery_manifest.to_csv(
    DISCOVERY_MANIFEST_PATH,
    index=False,
)

with open(
    DISCOVERY_MANIFEST_PATH,
    "rb",
) as f:
    discovery_hash = (
        hashlib.sha256(
            f.read()
        )
        .hexdigest()
    )

run_material = (
    PHASE7_VERSION
    + "|"
    + MODEL_ID
    + "|"
    + discovery_hash
)

RUN_ID = hashlib.sha256(
    run_material.encode()
).hexdigest()[:16]

print(
    "discovery SHA256:",
    discovery_hash,
)
print(
    "RUN_ID:",
    RUN_ID,
)

display(
    discovery_manifest.head()
)

discovery SHA256: 92b6e238fb8f5d7fb7a82cf6b3c098470c661db5b4a0784d0675a84b400f0b5c
RUN_ID: 8e9d8a0974b9fa94


,case_id,source_case_id,dataset_index,subject,prompt,true_text,new_text,D_clean_reference,TE_reference,D_clean_discovery,D0_discovery,TE_discovery
0,0,253,1517,Julien Courbet,Julien Courbet is a native speaker of,French,Russian,3.595154,2.537842,3.595154,0.689819,2.905334
1,1,9810,366,Windows Mobile 6.5,Windows Mobile 6.5 was developed by,Microsoft,Apple,1.853027,1.110779,1.853027,0.543274,1.309753
2,2,2538,600,Brisbane Airport,"Brisbane Airport, which was called after",Brisbane,Thor,4.434265,4.155151,4.434265,-0.989136,5.423401
3,3,12236,197,Francis X. DiLorenzo,Francis X. DiLorenzo has the position of,bishop,pope,2.203430,2.319031,2.203430,1.538025,0.665405
4,4,3244,1276,Nazi Party,Nazi Party is based in,Munich,Helsinki,1.562500,1.217224,1.562500,-1.202454,2.764954


# 5. Sites and activation patching

For every Transformer layer, two sites are evaluated:

- `S`: last token of the subject span;
- `Q`: final query token.

With 12 Pythia layers:

\[
24 \text{ sites}
\]

and

\[
\binom{24}{2}=276
\]

distinct pairs.

In [11]:
SITES = [
    (layer, role)
    for layer in range(N_LAYERS)
    for role in ["S", "Q"]
]

N_SITES = len(SITES)
N_PAIRS = (
    N_SITES * (N_SITES - 1) // 2
)

print("sites:", N_SITES)
print("pairs:", N_PAIRS)

sites: 24
pairs: 276


In [12]:
def patched_margin(
    case,
    clean_states,
    corrupt_emb,
    sites,
):
    patches_by_layer = {}

    for layer, role in sites:
        patches_by_layer.setdefault(
            layer,
            [],
        ).append(role)

    handles = []

    for layer, roles in (
        patches_by_layer.items()
    ):
        def make_hook(
            layer_idx,
            roles_here,
        ):
            def hook(
                module,
                inputs,
                output,
            ):
                if isinstance(
                    output,
                    tuple,
                ):
                    hidden = (
                        output[0]
                        .clone()
                    )
                else:
                    hidden = (
                        output.clone()
                    )

                for role in roles_here:
                    if role == "S":
                        pos = (
                            case[
                                "subject_last"
                            ]
                        )
                    elif role == "Q":
                        pos = (
                            case[
                                "query_pos"
                            ]
                        )
                    else:
                        raise ValueError(
                            role
                        )

                    clean_value = (
                        clean_states[
                            layer_idx + 1
                        ][:, pos, :]
                    )

                    hidden[
                        :, pos, :
                    ] = clean_value

                if isinstance(
                    output,
                    tuple,
                ):
                    return (
                        hidden,
                        *output[1:],
                    )

                return hidden

            return hook

        handle = (
            model.gpt_neox
            .layers[layer]
            .register_forward_hook(
                make_hook(
                    layer,
                    tuple(roles),
                )
            )
        )

        handles.append(
            handle
        )

    try:
        with torch.inference_mode():
            out = model(
                inputs_embeds=
                    corrupt_emb,

                attention_mask=
                    case[
                        "attention_mask"
                    ],
            )

        return logit_margin(
            out.logits,
            case["true_id"],
            case["new_id"],
        )

    finally:
        for handle in handles:
            handle.remove()

## Sanity check

The final-query patch should normally move the corrupted margin toward the clean margin for at least the demonstration case. This is a diagnostic, not a formal acceptance criterion.

In [13]:
case = discovery_cases[0]

(
    clean_states,
    corrupt_emb,
    D_clean,
    D0,
    TE,
) = discovery_runtime_state(
    case,
    need_hidden_states=True,
)

D_last_q = patched_margin(
    case,
    clean_states,
    corrupt_emb,
    [
        (N_LAYERS - 1, "Q")
    ],
)

print("subject:", case["subject"])
print("D clean:", D_clean)
print("D corrupt:", D0)
print("TE:", TE)
print("D last-Q patch:", D_last_q)

subject: Julien Courbet
D clean: 3.59515380859375
D corrupt: 0.6898193359375
TE: 2.90533447265625
D last-Q patch: 3.19195556640625


# 6. Discovery scan

For a single site \(i\),

\[
IE_i
=
\frac{D_i-D_0}{TE}.
\]

For a pair \(i,j\),

\[
R_{ij}
=
\frac{D_{ij}-D_0}{TE}.
\]

The normalized interaction term is

\[
J_{ij}
=
\frac{
D_{ij}-D_i-D_j+D_0
}{TE}.
\]

There is **one deterministic discovery corruption per case**. Therefore the exact expected discovery row counts are:

\[
32\times24=768
\]

single rows, and

\[
32\times276=8832
\]

pair rows.

In [14]:
SINGLE_PATH = (
    f"phase7_discovery_single_{RUN_ID}.csv"
)

PAIR_PATH = (
    f"phase7_discovery_pair_{RUN_ID}.csv"
)

single_columns = [
    "run_id",
    "case_id",
    "source_case_id",
    "layer",
    "role",
    "D_clean",
    "D0",
    "TE",
    "IE_norm",
]

pair_columns = [
    "run_id",
    "case_id",
    "source_case_id",
    "li",
    "ri",
    "lj",
    "rj",
    "J_norm",
    "pair_recovery",
]


def load_run_file(
    path,
    expected_columns,
):
    if not os.path.exists(path):
        return pd.DataFrame(
            columns=expected_columns
        )

    df = pd.read_csv(path)

    if len(df):
        if "run_id" not in df:
            raise RuntimeError(
                f"{path} has no run_id."
            )

        ids = set(
            df["run_id"]
            .astype(str)
        )

        if ids != {RUN_ID}:
            raise RuntimeError(
                f"{path} belongs to "
                f"another run: {ids}"
            )

    return df


single_df = load_run_file(
    SINGLE_PATH,
    single_columns,
)

pair_df = load_run_file(
    PAIR_PATH,
    pair_columns,
)

print(
    "existing single rows:",
    len(single_df),
)
print(
    "existing pair rows:",
    len(pair_df),
)

existing single rows: 0
existing pair rows: 0


In [15]:
single_key = [
    "case_id",
    "layer",
    "role",
]

pair_key = [
    "case_id",
    "li",
    "ri",
    "lj",
    "rj",
]


def complete_case_ids(
    single_df,
    pair_df,
):
    if len(single_df):
        single_counts = (
            single_df
            .groupby("case_id")
            .size()
        )

        complete_single = set(
            single_counts[
                single_counts
                == N_SITES
            ].index
        )
    else:
        complete_single = set()

    if len(pair_df):
        pair_counts = (
            pair_df
            .groupby("case_id")
            .size()
        )

        complete_pair = set(
            pair_counts[
                pair_counts
                == N_PAIRS
            ].index
        )
    else:
        complete_pair = set()

    return (
        complete_single
        & complete_pair
    )


if single_df.duplicated(
    single_key
).any():
    raise RuntimeError(
        "Duplicate discovery single keys "
        "found. Remove only this run's "
        "result file and rerun."
    )

if pair_df.duplicated(
    pair_key
).any():
    raise RuntimeError(
        "Duplicate discovery pair keys "
        "found. Remove only this run's "
        "result file and rerun."
    )

complete_cases = complete_case_ids(
    single_df,
    pair_df,
)

print(
    "already complete:",
    len(complete_cases),
)

already complete: 0


In [16]:
single_rows = (
    single_df.to_dict("records")
    if len(single_df)
    else []
)

pair_rows = (
    pair_df.to_dict("records")
    if len(pair_df)
    else []
)

for ci, case in enumerate(
    discovery_cases
):
    case_id = case["case_id"]

    if case_id in complete_cases:
        print(
            f"{ci+1}/{N_DISCOVERY}",
            case["subject"],
            "| already complete",
        )
        continue

    (
        clean_states,
        corrupt_emb,
        D_clean,
        D0,
        TE,
    ) = discovery_runtime_state(
        case,
        need_hidden_states=True,
    )

    if TE <= MIN_DISCOVERY_TE:
        raise RuntimeError(
            "Frozen discovery case "
            f"{case_id} now has TE={TE}. "
            "This indicates environment/model "
            "drift relative to manifest creation."
        )

    single_D = {}
    new_single_rows = []

    for site in SITES:
        Di = patched_margin(
            case,
            clean_states,
            corrupt_emb,
            [site],
        )

        single_D[site] = Di

        IE_norm = (
            Di - D0
        ) / TE

        new_single_rows.append({
            "run_id":
                RUN_ID,

            "case_id":
                case_id,

            "source_case_id":
                case[
                    "source_case_id"
                ],

            "layer":
                site[0],

            "role":
                site[1],

            "D_clean":
                D_clean,

            "D0":
                D0,

            "TE":
                TE,

            "IE_norm":
                IE_norm,
        })

    new_pair_rows = []

    for i in range(N_SITES):
        for j in range(
            i + 1,
            N_SITES,
        ):
            si = SITES[i]
            sj = SITES[j]

            Dij = patched_margin(
                case,
                clean_states,
                corrupt_emb,
                [si, sj],
            )

            Di = single_D[si]
            Dj = single_D[sj]

            J_norm = (
                Dij
                - Di
                - Dj
                + D0
            ) / TE

            pair_recovery = (
                Dij - D0
            ) / TE

            new_pair_rows.append({
                "run_id":
                    RUN_ID,

                "case_id":
                    case_id,

                "source_case_id":
                    case[
                        "source_case_id"
                    ],

                "li":
                    si[0],

                "ri":
                    si[1],

                "lj":
                    sj[0],

                "rj":
                    sj[1],

                "J_norm":
                    J_norm,

                "pair_recovery":
                    pair_recovery,
            })

    # Replace a whole case atomically in memory.
    # Partial rows from a previous interrupted
    # attempt are never retained.
    single_rows = [
        r for r in single_rows
        if int(r["case_id"])
        != case_id
    ]

    pair_rows = [
        r for r in pair_rows
        if int(r["case_id"])
        != case_id
    ]

    single_rows.extend(
        new_single_rows
    )
    pair_rows.extend(
        new_pair_rows
    )

    single_df = pd.DataFrame(
        single_rows,
        columns=single_columns,
    )

    pair_df = pd.DataFrame(
        pair_rows,
        columns=pair_columns,
    )

    single_df.to_csv(
        SINGLE_PATH,
        index=False,
    )

    pair_df.to_csv(
        PAIR_PATH,
        index=False,
    )

    print(
        f"{ci+1}/{N_DISCOVERY}",
        case["subject"],
        "| TE =",
        round(TE, 3),
    )

1/32 Julien Courbet | TE = 2.905
2/32 Windows Mobile 6.5 | TE = 1.31
3/32 Brisbane Airport | TE = 5.423
4/32 Francis X. DiLorenzo | TE = 0.665
5/32 Nazi Party | TE = 2.765
6/32 Galina Starovoytova | TE = 1.861
7/32 Charles-Joseph Panckoucke | TE = 1.534
8/32 Saint-Jacques-de-la-Lande | TE = 4.283
9/32 The Kathmandu Post | TE = 1.633
10/32 Virginia | TE = 2.006
11/32 Broadway Open House | TE = 1.605
12/32 Microsoft Project | TE = 0.884
13/32 R. H. Bruce Lockhart | TE = 2.153
14/32 Honolulu International Airport | TE = 2.534
15/32 Hurrem sultan | TE = 1.541
16/32 New Zealand Labour Party | TE = 5.625
17/32 1977 Moscow bombings | TE = 3.124
18/32 Sommariva del Bosco | TE = 1.105
19/32 Google Person Finder | TE = 4.47
20/32 New Club, Edinburgh | TE = 5.37
21/32 Boeing Defense, Space & Security | TE = 2.355
22/32 Thomas Beaufort, Duke of Exeter | TE = 1.629
23/32 Michelangelo | TE = 1.824
24/32 Jerusalem syndrome | TE = 4.874
25/32 Bobby Taylor & the Vancouvers | TE = 0.824
26/32 Nicholas F

# 7. Discovery integrity checks

No `drop_duplicates()` is used to hide inconsistencies.

The scan is accepted only if it has the exact expected cardinality and unique keys.

In [17]:
single_df = pd.read_csv(
    SINGLE_PATH
)
pair_df = pd.read_csv(
    PAIR_PATH
)

EXPECTED_SINGLE = (
    N_DISCOVERY
    * N_SITES
)

EXPECTED_PAIR = (
    N_DISCOVERY
    * N_PAIRS
)

print(
    "single rows:",
    len(single_df),
    "/",
    EXPECTED_SINGLE,
)

print(
    "pair rows:",
    len(pair_df),
    "/",
    EXPECTED_PAIR,
)

print(
    "single cases:",
    single_df["case_id"]
    .nunique(),
)

print(
    "pair cases:",
    pair_df["case_id"]
    .nunique(),
)

assert len(single_df) == EXPECTED_SINGLE
assert len(pair_df) == EXPECTED_PAIR

assert not single_df.duplicated(
    single_key
).any()

assert not pair_df.duplicated(
    pair_key
).any()

assert set(
    single_df["case_id"]
) == set(range(N_DISCOVERY))

assert set(
    pair_df["case_id"]
) == set(range(N_DISCOVERY))

assert set(
    single_df["run_id"]
    .astype(str)
) == {RUN_ID}

assert set(
    pair_df["run_id"]
    .astype(str)
) == {RUN_ID}

print("Discovery integrity: OK")

single rows: 768 / 768
pair rows: 8832 / 8832
single cases: 32
pair cases: 32
Discovery integrity: OK


# 8. Discovery statistics and frozen pair selection

## AIE pair

Rank single sites by

\[
\mathrm{AIE}(i)
=
\mathbb E[IE_i].
\]

## Recovery-optimal pair

Rank pairs by

\[
\mathbb E[R_{ij}].
\]

This is the pair used in the **primary holdout comparison** because the endpoint itself is causal recovery.

## Synergy-optimal pair

Separately rank by

\[
\mathbb E[J_{ij}].
\]

This is reported as a mechanistic diagnostic. A pair can have extremely high recovery while having \(J<0\), which indicates overlapping/redundant causal contribution rather than positive synergy.

In [18]:
site_stats = (
    single_df
    .groupby(
        ["layer", "role"]
    )["IE_norm"]
    .agg(
        mean="mean",
        std="std",
        count="count",
    )
    .reset_index()
    .sort_values(
        "mean",
        ascending=False,
    )
)

display(
    site_stats.head(15)
)

assert (
    site_stats["count"]
    == N_DISCOVERY
).all()

,layer,role,mean,std,count
20,10,Q,0.987370,0.045189,32
18,9,Q,0.960867,0.075076,32
16,8,Q,0.716684,0.197244,32
22,11,Q,0.575196,0.365307,32
9,4,S,0.472561,0.478504,32
7,3,S,0.462277,0.480814,32
5,2,S,0.440523,0.399043,32
11,5,S,0.412775,0.444306,32
13,6,S,0.410424,0.450634,32
15,7,S,0.382859,0.417760,32


In [19]:
pair_stats = (
    pair_df
    .groupby(
        [
            "li",
            "ri",
            "lj",
            "rj",
        ]
    )
    .agg(
        mean_J=(
            "J_norm",
            "mean",
        ),

        std_J=(
            "J_norm",
            "std",
        ),

        mean_pair_recovery=(
            "pair_recovery",
            "mean",
        ),

        std_pair_recovery=(
            "pair_recovery",
            "std",
        ),

        count=(
            "pair_recovery",
            "count",
        ),
    )
    .reset_index()
)

assert len(pair_stats) == N_PAIRS
assert (
    pair_stats["count"]
    == N_DISCOVERY
).all()

print("Top synergy:")
display(
    pair_stats
    .sort_values(
        "mean_J",
        ascending=False,
    )
    .head(15)
)

print("Top recovery:")
display(
    pair_stats
    .sort_values(
        "mean_pair_recovery",
        ascending=False,
    )
    .head(15)
)

Top synergy:


,li,ri,lj,rj,mean_J,std_J,mean_pair_recovery,std_pair_recovery,count
186,5,Q,6,S,0.046187,0.149108,0.649358,0.414090,32
188,5,Q,7,S,0.038853,0.142868,0.614460,0.401248,32
197,5,S,5,Q,0.037137,0.151005,0.642659,0.408790,32
211,6,Q,7,S,0.033969,0.142351,0.653233,0.415945,32
161,4,Q,7,S,0.029494,0.133082,0.508269,0.393364,32
159,4,Q,6,S,0.029014,0.147948,0.535354,0.414680,32
220,6,S,6,Q,0.027068,0.147956,0.673898,0.429384,32
239,7,S,7,Q,0.022935,0.140439,0.745350,0.395250,32
128,3,Q,6,S,0.022759,0.138433,0.470217,0.423450,32
157,4,Q,5,S,0.022436,0.152652,0.531128,0.402537,32


Top recovery:


,li,ri,lj,rj,mean_J,std_J,mean_pair_recovery,std_pair_recovery,count
181,4,S,10,Q,-0.461625,0.462857,0.998305,0.042915,32
206,5,S,10,Q,-0.402876,0.430041,0.997269,0.044453,32
152,3,S,10,Q,-0.453979,0.470104,0.995668,0.045094,32
244,7,S,10,Q,-0.375703,0.408988,0.994526,0.040620,32
227,6,S,10,Q,-0.403666,0.434197,0.994128,0.045896,32
257,8,S,10,Q,-0.220521,0.223208,0.993638,0.039244,32
119,2,S,10,Q,-0.435327,0.393665,0.992566,0.047369,32
41,0,S,10,Q,-0.323816,0.356939,0.991986,0.050057,32
272,10,S,10,Q,0.003343,0.010058,0.991912,0.044135,32
266,9,S,10,Q,-0.007745,0.022368,0.991809,0.043907,32


In [20]:
AIE_PAIR = [
    (
        int(row.layer),
        str(row.role),
    )
    for row in (
        site_stats
        .head(2)
        .itertuples()
    )
]

recovery_best = (
    pair_stats
    .sort_values(
        "mean_pair_recovery",
        ascending=False,
    )
    .iloc[0]
)

RECOVERY_PAIR = [
    (
        int(
            recovery_best["li"]
        ),
        str(
            recovery_best["ri"]
        ),
    ),
    (
        int(
            recovery_best["lj"]
        ),
        str(
            recovery_best["rj"]
        ),
    ),
]

synergy_best = (
    pair_stats
    .sort_values(
        "mean_J",
        ascending=False,
    )
    .iloc[0]
)

SYNERGY_PAIR = [
    (
        int(
            synergy_best["li"]
        ),
        str(
            synergy_best["ri"]
        ),
    ),
    (
        int(
            synergy_best["lj"]
        ),
        str(
            synergy_best["rj"]
        ),
    ),
]

print("AIE pair:", AIE_PAIR)

print(
    "Recovery-optimal pair:",
    RECOVERY_PAIR,
)

print(
    "Recovery-optimal mean recovery:",
    float(
        recovery_best[
            "mean_pair_recovery"
        ]
    ),
)

print(
    "Recovery-optimal mean J:",
    float(
        recovery_best[
            "mean_J"
        ]
    ),
)

print(
    "Synergy-optimal pair:",
    SYNERGY_PAIR,
)

print(
    "Synergy-optimal mean J:",
    float(
        synergy_best[
            "mean_J"
        ]
    ),
)

print(
    "Synergy-optimal mean recovery:",
    float(
        synergy_best[
            "mean_pair_recovery"
        ]
    ),
)

AIE pair: [(10, 'Q'), (9, 'Q')]
Recovery-optimal pair: [(4, 'S'), (10, 'Q')]
Recovery-optimal mean recovery: 0.9983053512677614
Recovery-optimal mean J: -0.461625213221331
Synergy-optimal pair: [(5, 'Q'), (6, 'S')]
Synergy-optimal mean J: 0.046186691589701234
Synergy-optimal mean recovery: 0.6493579901271037


## Discovery comparison at equal budget

This comparison is descriptive only. The holdout remains the confirmatory test.

In [21]:
def get_pair_rows(
    df,
    pair,
):
    a, b = sorted(pair)

    mask = (
        (df["li"] == a[0])
        &
        (df["ri"] == a[1])
        &
        (df["lj"] == b[0])
        &
        (df["rj"] == b[1])
    )

    return df[
        mask
    ].copy()


aie_disc = get_pair_rows(
    pair_df,
    AIE_PAIR,
)

recovery_disc = get_pair_rows(
    pair_df,
    RECOVERY_PAIR,
)

discovery_compare = (
    aie_disc[
        [
            "case_id",
            "pair_recovery",
        ]
    ]
    .rename(
        columns={
            "pair_recovery":
                "recovery_AIE"
        }
    )
    .merge(
        recovery_disc[
            [
                "case_id",
                "pair_recovery",
            ]
        ]
        .rename(
            columns={
                "pair_recovery":
                    "recovery_RECOVERY"
            }
        ),
        on="case_id",
        how="inner",
        validate="one_to_one",
    )
)

discovery_compare["delta"] = (
    discovery_compare[
        "recovery_RECOVERY"
    ]
    -
    discovery_compare[
        "recovery_AIE"
    ]
)

display(
    discovery_compare.describe()
)

print(
    "Discovery mean delta:",
    discovery_compare[
        "delta"
    ].mean(),
)

,case_id,recovery_AIE,recovery_RECOVERY,delta
count,32.000000,32.000000,32.000000,32.000000
mean,15.500000,0.987370,0.998305,0.010935
std,9.380832,0.045189,0.042915,0.032425
min,0.000000,0.831117,0.916460,-0.018738
25%,7.750000,0.973618,0.982679,-0.001855
50%,15.500000,0.992364,0.994323,0.000163
75%,23.250000,1.007257,1.008376,0.005363
max,31.000000,1.101495,1.145780,0.131261


Discovery mean delta: 0.010935391136973064


# 9. Freeze the discovery selection

After this cell, no discovery statistic is used to alter the primary holdout rule.

In [22]:
SELECTION_PATH = (
    "phase7_frozen_selection_clean.json"
)

phase7_selection = {
    "phase7_version":
        PHASE7_VERSION,

    "run_id":
        RUN_ID,

    "model":
        MODEL_ID,

    "discovery_n":
        N_DISCOVERY,

    "discovery_manifest_sha256":
        discovery_hash,

    "AIE_pair":
        AIE_PAIR,

    "recovery_optimal_pair":
        RECOVERY_PAIR,

    "synergy_optimal_pair_descriptive":
        SYNERGY_PAIR,

    "selection_rule_AIE":
        (
            "top two sites by mean "
            "normalized single-site IE"
        ),

    "selection_rule_primary_pair":
        (
            "pair with maximum mean "
            "measured pair recovery"
        ),

    "synergy_diagnostic":
        (
            "pair with maximum mean "
            "normalized J; descriptive only"
        ),

    "primary_holdout_metric":
        "normalized causal recovery",

    "primary_success":
        (
            "paired bootstrap 95% CI "
            "lower bound > 0"
        ),

    "practical_threshold":
        PRACTICAL_THRESHOLD,

    "noise_scales":
        NOISE_SCALES,

    "noise_seeds":
        NOISE_SEEDS,

    "primary_scale":
        PRIMARY_SCALE,
}

with open(
    SELECTION_PATH,
    "w",
) as f:
    json.dump(
        phase7_selection,
        f,
        indent=2,
    )

print(
    "Frozen selection written to:",
    SELECTION_PATH,
)

Frozen selection written to: phase7_frozen_selection_clean.json


# 10. Build a 128-case subject-disjoint holdout

The holdout excludes:

\[
\text{Phase 5}
\cup
\text{Phase 6A}
\cup
\text{Phase 6B}
\cup
\text{Phase 7 discovery}.
\]

Only the **reference** preparation criteria are used to select holdout cases. Evaluation corruption validity is handled later per noise condition via `MIN_EVAL_TE`.

In [23]:
DISCOVERY_SUBJECTS = {
    norm_subject(c["subject"])
    for c in discovery_cases
}

ALL_SEEN_SUBJECTS = (
    PREVIOUS_SUBJECTS
    | DISCOVERY_SUBJECTS
)

rng_holdout = (
    np.random.default_rng(
        HOLDOUT_SEED
    )
)

indices_holdout = (
    rng_holdout.permutation(
        len(counterfact)
    )
)

holdout_cases = []
holdout_subjects = set()

for idx in indices_holdout:
    record = counterfact[
        int(idx)
    ]

    subject_key = norm_subject(
        record[
            "requested_rewrite"
        ]["subject"]
    )

    if subject_key in ALL_SEEN_SUBJECTS:
        continue

    if subject_key in holdout_subjects:
        continue

    case = prepare_case(
        record,
        int(idx),
    )

    if case is None:
        continue

    holdout_cases.append(case)
    holdout_subjects.add(
        subject_key
    )

    if len(holdout_cases) >= N_HOLDOUT:
        break


if len(holdout_cases) != N_HOLDOUT:
    raise RuntimeError(
        f"Only found {len(holdout_cases)} "
        f"holdout cases; "
        f"expected {N_HOLDOUT}."
    )

for case_id, case in enumerate(
    holdout_cases
):
    case["case_id"] = int(case_id)

assert len(
    holdout_subjects
) == N_HOLDOUT

assert (
    holdout_subjects
    & ALL_SEEN_SUBJECTS
) == set()

print(
    "holdout:",
    len(holdout_cases),
)

print(
    "overlap with previous/discovery:",
    len(
        holdout_subjects
        & ALL_SEEN_SUBJECTS
    ),
)

holdout: 128
overlap with previous/discovery: 0


In [24]:
HOLDOUT_MANIFEST_PATH = (
    "phase7_holdout_manifest_clean.csv"
)

holdout_manifest = pd.DataFrame([
    {
        "case_id":
            c["case_id"],

        "source_case_id":
            c["source_case_id"],

        "dataset_index":
            c["dataset_index"],

        "subject":
            c["subject"],

        "prompt":
            c["prompt"],

        "true_text":
            c["true_text"],

        "new_text":
            c["new_text"],

        "D_clean_reference":
            c["D_clean_reference"],

        "TE_reference":
            c["TE_reference"],
    }
    for c in holdout_cases
])

assert len(
    holdout_manifest
) == N_HOLDOUT

assert holdout_manifest[
    "case_id"
].is_unique

assert holdout_manifest[
    "subject"
].map(
    norm_subject
).is_unique

holdout_manifest.to_csv(
    HOLDOUT_MANIFEST_PATH,
    index=False,
)

with open(
    HOLDOUT_MANIFEST_PATH,
    "rb",
) as f:
    HOLDOUT_HASH = (
        hashlib.sha256(
            f.read()
        )
        .hexdigest()
    )

print(
    "holdout SHA256:",
    HOLDOUT_HASH,
)

display(
    holdout_manifest.head()
)

holdout SHA256: 280139f3d160626accc5332f5c2e4dc82f0592e8dc3852acca7edd58d6038f1b


,case_id,source_case_id,dataset_index,subject,prompt,true_text,new_text,D_clean_reference,TE_reference
0,0,13107,1893,Harry Welles Rusk,Harry Welles Rusk died at,Baltimore,Lisbon,2.230591,2.908875
1,1,10893,1059,Jacques Attali,The mother tongue of Jacques Attali is,French,Chinese,1.924805,1.171326
2,2,16180,938,Northlands,"Northlands, whose headquarters are in",Edmonton,Amsterdam,2.130005,2.127075
3,3,3779,462,Chevrolet Chevette,Chevrolet Chevette is a product of,Chevrolet,IBM,3.439758,7.766113
4,4,5525,2051,Charles Kennedy,Charles Kennedy worked in the city of,London,Brisbane,5.279297,1.135376


# 11. Freeze the holdout protocol

In [25]:
PROTOCOL_PATH = (
    "phase7_frozen_protocol_clean.json"
)

phase7_protocol = {
    "phase7_version":
        PHASE7_VERSION,

    "discovery_run_id":
        RUN_ID,

    "experiment":
        (
            "Phase 7 clean Pythia "
            "cross-model replication"
        ),

    "model":
        MODEL_ID,

    "discovery_n":
        N_DISCOVERY,

    "holdout_n":
        N_HOLDOUT,

    "AIE_pair":
        AIE_PAIR,

    "recovery_optimal_pair":
        RECOVERY_PAIR,

    "synergy_optimal_pair_descriptive":
        SYNERGY_PAIR,

    "holdout_manifest_sha256":
        HOLDOUT_HASH,

    "primary_scale":
        PRIMARY_SCALE,

    "noise_scales":
        NOISE_SCALES,

    "noise_seeds":
        NOISE_SEEDS,

    "min_eval_TE":
        MIN_EVAL_TE,

    "primary_metric":
        "normalized causal recovery",

    "primary_delta":
        (
            "recovery_RECOVERY "
            "- recovery_AIE"
        ),

    "primary_success":
        (
            "paired bootstrap 95% CI "
            "lower bound > 0"
        ),

    "practical_threshold":
        PRACTICAL_THRESHOLD,
}

with open(
    PROTOCOL_PATH,
    "w",
) as f:
    json.dump(
        phase7_protocol,
        f,
        indent=2,
    )

print(
    "Frozen protocol:",
    PROTOCOL_PATH,
)

Frozen protocol: phase7_frozen_protocol_clean.json


# 12. Holdout evaluation

For each holdout case, noise scale and noise seed:

\[
TE
=
D_{\mathrm{clean}}
-
D_{\mathrm{corr}}.
\]

A condition is evaluable only if

\[
TE\ge
\texttt{MIN_EVAL_TE}.
\]

Then:

\[
R_{\mathrm{AIE}}
=
\frac{
D_{\mathrm{AIE}}
-
D_{\mathrm{corr}}
}{TE}
\]

and

\[
R_{\mathrm{RECOVERY}}
=
\frac{
D_{\mathrm{RECOVERY}}
-
D_{\mathrm{corr}}
}{TE}.
\]

Finally:

\[
\Delta
=
R_{\mathrm{RECOVERY}}
-
R_{\mathrm{AIE}}.
\]

In [26]:
def evaluate_holdout_condition(
    case,
    noise_multiplier,
    noise_seed,
):
    with torch.inference_mode():
        clean_out = model(
            input_ids=
                case["input_ids"],

            attention_mask=
                case[
                    "attention_mask"
                ],

            output_hidden_states=True,
        )

    clean_states = (
        clean_out.hidden_states
    )

    D_clean = logit_margin(
        clean_out.logits,
        case["true_id"],
        case["new_id"],
    )

    actual_seed = (
        int(noise_seed)
        +
        int(
            case[
                "source_case_id"
            ]
        )
    )

    corrupt_emb = (
        corrupted_embeddings(
            case["input_ids"],
            case[
                "subject_positions"
            ],
            actual_seed,
            noise_multiplier,
        )
    )

    with torch.inference_mode():
        corrupt_out = model(
            inputs_embeds=
                corrupt_emb,

            attention_mask=
                case[
                    "attention_mask"
                ],
        )

    D_corr = logit_margin(
        corrupt_out.logits,
        case["true_id"],
        case["new_id"],
    )

    TE = D_clean - D_corr

    D_AIE = patched_margin(
        case,
        clean_states,
        corrupt_emb,
        AIE_PAIR,
    )

    D_RECOVERY = patched_margin(
        case,
        clean_states,
        corrupt_emb,
        RECOVERY_PAIR,
    )

    valid = bool(
        TE >= MIN_EVAL_TE
    )

    if valid:
        recovery_AIE = (
            D_AIE - D_corr
        ) / TE

        recovery_RECOVERY = (
            D_RECOVERY - D_corr
        ) / TE

        delta = (
            recovery_RECOVERY
            - recovery_AIE
        )
    else:
        recovery_AIE = np.nan
        recovery_RECOVERY = np.nan
        delta = np.nan

    return {
        "case_id":
            case["case_id"],

        "source_case_id":
            case[
                "source_case_id"
            ],

        "noise_scale":
            float(
                noise_multiplier
            ),

        "noise_seed":
            int(noise_seed),

        "D_clean":
            D_clean,

        "D_corr":
            D_corr,

        "TE":
            TE,

        "valid":
            valid,

        "D_AIE":
            D_AIE,

        "D_RECOVERY":
            D_RECOVERY,

        "recovery_AIE":
            recovery_AIE,

        "recovery_RECOVERY":
            recovery_RECOVERY,

        "delta_recovery":
            delta,

        "margin_advantage_RECOVERY":
            (
                D_RECOVERY
                - D_AIE
            ),

        "correct_AIE":
            float(
                D_AIE > 0
            ),

        "correct_RECOVERY":
            float(
                D_RECOVERY > 0
            ),

        "holdout_manifest_sha256":
            HOLDOUT_HASH,

        "discovery_run_id":
            RUN_ID,
    }

In [27]:
RESULT_PATH = (
    f"phase7_holdout_results_{RUN_ID}.csv"
)

result_key = [
    "case_id",
    "noise_scale",
    "noise_seed",
]

if os.path.exists(
    RESULT_PATH
):
    phase7_results = pd.read_csv(
        RESULT_PATH
    )

    if len(
        phase7_results
    ):
        hashes = set(
            phase7_results[
                "holdout_manifest_sha256"
            ]
            .dropna()
            .astype(str)
        )

        run_ids = set(
            phase7_results[
                "discovery_run_id"
            ]
            .dropna()
            .astype(str)
        )

        if hashes != {
            HOLDOUT_HASH
        }:
            raise RuntimeError(
                "Holdout result file "
                "belongs to another manifest."
            )

        if run_ids != {
            RUN_ID
        }:
            raise RuntimeError(
                "Holdout result file "
                "belongs to another "
                "discovery selection."
            )

        if phase7_results.duplicated(
            result_key
        ).any():
            raise RuntimeError(
                "Duplicate holdout result keys."
            )
else:
    phase7_results = (
        pd.DataFrame()
    )

done = set()

if len(
    phase7_results
):
    done = set(
        zip(
            phase7_results[
                "case_id"
            ].astype(int),

            phase7_results[
                "noise_scale"
            ].astype(float),

            phase7_results[
                "noise_seed"
            ].astype(int),
        )
    )

print(
    "already complete:",
    len(done),
)

already complete: 0


In [28]:
rows = (
    phase7_results
    .to_dict("records")
    if len(phase7_results)
    else []
)

for ci, case in enumerate(
    holdout_cases
):
    for scale in NOISE_SCALES:
        for seed in NOISE_SEEDS:
            key = (
                int(
                    case["case_id"]
                ),
                float(scale),
                int(seed),
            )

            if key in done:
                continue

            row = (
                evaluate_holdout_condition(
                    case,
                    scale,
                    seed,
                )
            )

            rows.append(row)
            done.add(key)

            pd.DataFrame(
                rows
            ).to_csv(
                RESULT_PATH,
                index=False,
            )

    print(
        f"{ci+1}/{N_HOLDOUT}",
        case["subject"],
    )

phase7_results = pd.DataFrame(
    rows
)

1/128 Harry Welles Rusk
2/128 Jacques Attali
3/128 Northlands
4/128 Chevrolet Chevette
5/128 Charles Kennedy
6/128 Guinea-Bissau
7/128 Orikhiv
8/128 IBM DeveloperWorks
9/128 Wiener AC
10/128 Sakichi Toyoda
11/128 Syed Kalbe Hussain
12/128 Fyodor Pavlovich Reshetnikov
13/128 Molina de Segura
14/128 Chardonne
15/128 Sky Cinema
16/128 Firuz Shah Tughlaq
17/128 LaFerrari
18/128 Alex Rodriguez
19/128 Fernand Ledoux
20/128 Palace Software
21/128 68th Venice International Film Festival
22/128 Ernst Reuter
23/128 Nelson Valdez
24/128 Wiener schnitzel
25/128 La Revista Blanca
26/128 Charles Dupin
27/128 Veysian Rural District
28/128 Iceland
29/128 Ginette Garcin
30/128 Roman Empire
31/128 Hussniya Jabara
32/128 penne
33/128 Ferrari 575M Maranello
34/128 Pierre Lescure
35/128 World Association of Girl Guides and Girl Scouts
36/128 Augustin Barruel
37/128 Jeff Tweedy
38/128 Visual Basic .NET
39/128 Eureka Brass Band
40/128 Belarus
41/128 Operation Apollo
42/128 Wear OS
43/128 Brickell Key
44/128 

# 13. Holdout integrity

In [29]:
EXPECTED_HOLDOUT_ROWS = (
    N_HOLDOUT
    * len(NOISE_SCALES)
    * len(NOISE_SEEDS)
)

print(
    "holdout rows:",
    len(phase7_results),
    "/",
    EXPECTED_HOLDOUT_ROWS,
)

assert (
    len(phase7_results)
    == EXPECTED_HOLDOUT_ROWS
)

assert not phase7_results.duplicated(
    result_key
).any()

assert set(
    phase7_results[
        "case_id"
    ].astype(int)
) == set(range(N_HOLDOUT))

print(
    "valid fraction:",
    phase7_results[
        "valid"
    ].mean(),
)

print("Holdout integrity: OK")

holdout rows: 1152 / 1152
valid fraction: 0.8854166666666666
Holdout integrity: OK


# 14. Primary case-level analysis

The bootstrap unit is the **case**, not the individual noise draw.

At the primary scale \(3.0\):

1. average valid noise-seed results within each case;
2. bootstrap those case-level deltas.

This preserves the original Phase-7 logic while preventing pseudo-replication over noise seeds.

In [30]:
primary = phase7_results[
    (
        phase7_results[
            "noise_scale"
        ] == PRIMARY_SCALE
    )
    &
    (
        phase7_results[
            "valid"
        ]
    )
].copy()

primary_case = (
    primary
    .groupby("case_id")
    .agg(
        recovery_AIE=(
            "recovery_AIE",
            "mean",
        ),

        recovery_RECOVERY=(
            "recovery_RECOVERY",
            "mean",
        ),

        delta=(
            "delta_recovery",
            "mean",
        ),

        margin_advantage=(
            "margin_advantage_RECOVERY",
            "mean",
        ),

        n_valid_seeds=(
            "noise_seed",
            "nunique",
        ),
    )
)

display(
    primary_case.head()
)

print(
    "primary cases with >=1 valid seed:",
    len(primary_case),
)

print(
    "complete 3-seed primary cases:",
    int(
        (
            primary_case[
                "n_valid_seeds"
            ]
            == len(NOISE_SEEDS)
        ).sum()
    ),
)

,recovery_AIE,recovery_RECOVERY,delta,margin_advantage,n_valid_seeds
case_id,,,,,
0,0.915039,0.907999,-0.007040,-0.013346,3
1,0.994461,1.007300,0.012839,0.018921,3
2,1.020967,1.028933,0.007966,0.036458,3
3,0.983721,0.985188,0.001467,0.011475,3
4,0.971388,0.965785,-0.005603,-0.003133,3


primary cases with >=1 valid seed: 127
complete 3-seed primary cases: 98


In [31]:
def bootstrap_mean_ci(
    values,
    n_boot=10000,
    seed=12345,
):
    x = np.asarray(
        values,
        dtype=float,
    )

    x = x[
        np.isfinite(x)
    ]

    if len(x) == 0:
        return (
            np.nan,
            np.nan,
            np.nan,
        )

    rng = (
        np.random.default_rng(
            seed
        )
    )

    n = len(x)

    boot = np.empty(
        n_boot,
        dtype=float,
    )

    for b in range(
        n_boot
    ):
        sample = rng.choice(
            x,
            size=n,
            replace=True,
        )

        boot[b] = (
            sample.mean()
        )

    return (
        float(x.mean()),
        float(
            np.quantile(
                boot,
                0.025,
            )
        ),
        float(
            np.quantile(
                boot,
                0.975,
            )
        ),
    )

In [32]:
delta_mean, lo, hi = (
    bootstrap_mean_ci(
        primary_case[
            "delta"
        ]
    )
)

print(
    "n primary cases:",
    len(primary_case),
)

print(
    "AIE mean recovery:",
    primary_case[
        "recovery_AIE"
    ].mean(),
)

print(
    "Recovery-optimal pair mean recovery:",
    primary_case[
        "recovery_RECOVERY"
    ].mean(),
)

print(
    "delta:",
    delta_mean,
)

print(
    "95% CI:",
    (lo, hi),
)

print(
    "statistical success:",
    lo > 0,
)

print(
    "practical success:",
    delta_mean
    >= PRACTICAL_THRESHOLD,
)

print(
    "win rate:",
    (
        primary_case[
            "delta"
        ] > 0
    ).mean(),
)

print(
    "tie rate:",
    np.isclose(
        primary_case[
            "delta"
        ],
        0,
    ).mean(),
)

n primary cases: 127
AIE mean recovery: 0.9753848092692807
Recovery-optimal pair mean recovery: 0.9829514588334141
delta: 0.007566649564133287
95% CI: (0.00011758616999788715, 0.014898958112072785)
statistical success: True
practical success: False
win rate: 0.6377952755905512
tie rate: 0.007874015748031496


## Complete-seed sensitivity analysis

This is not a replacement for the frozen primary analysis. It asks whether the conclusion changes when retaining only cases for which all three primary-scale noise seeds satisfy the TE validity threshold.

In [33]:
complete_primary = (
    primary_case[
        primary_case[
            "n_valid_seeds"
        ] == len(NOISE_SEEDS)
    ]
)

complete_mean, complete_lo, complete_hi = (
    bootstrap_mean_ci(
        complete_primary[
            "delta"
        ],
        seed=54321,
    )
)

print(
    "complete cases:",
    len(complete_primary),
)

print(
    "complete-case delta:",
    complete_mean,
)

print(
    "complete-case 95% CI:",
    (
        complete_lo,
        complete_hi,
    ),
)

complete cases: 98
complete-case delta: 0.00831471191313748
complete-case 95% CI: (0.001453472796716561, 0.015504715807876846)


# 15. Noise-scale robustness

In [34]:
scale_case = (
    phase7_results[
        phase7_results[
            "valid"
        ]
    ]
    .groupby(
        [
            "noise_scale",
            "case_id",
        ]
    )
    .agg(
        delta=(
            "delta_recovery",
            "mean",
        ),

        recovery_AIE=(
            "recovery_AIE",
            "mean",
        ),

        recovery_RECOVERY=(
            "recovery_RECOVERY",
            "mean",
        ),
    )
    .reset_index()
)

scale_rows = []

for scale, d in (
    scale_case
    .groupby(
        "noise_scale"
    )
):
    mean, ci_lo, ci_hi = (
        bootstrap_mean_ci(
            d["delta"],
            seed=(
                10000
                +
                int(
                    round(
                        float(scale)
                        * 100
                    )
                )
            ),
        )
    )

    scale_rows.append({
        "noise_scale":
            float(scale),

        "n_cases":
            int(
                d[
                    "case_id"
                ].nunique()
            ),

        "mean_recovery_AIE":
            float(
                d[
                    "recovery_AIE"
                ].mean()
            ),

        "mean_recovery_RECOVERY":
            float(
                d[
                    "recovery_RECOVERY"
                ].mean()
            ),

        "mean_delta":
            mean,

        "ci_low":
            ci_lo,

        "ci_high":
            ci_hi,

        "win_rate":
            float(
                (
                    d[
                        "delta"
                    ] > 0
                ).mean()
            ),
    })

scale_summary = pd.DataFrame(
    scale_rows
)

display(
    scale_summary
)

,noise_scale,n_cases,mean_recovery_AIE,mean_recovery_RECOVERY,mean_delta,ci_low,ci_high,win_rate
0,2.5,125,0.971430,0.981211,0.009781,0.002744,0.017140,0.664000
1,3.0,127,0.975385,0.982951,0.007567,0.000167,0.014699,0.637795
2,3.5,126,0.975447,0.982388,0.006941,0.000842,0.013081,0.642857


# 16. Raw-margin sensitivity check

Normalized recovery can become noisy when \(TE\) is small. As a secondary diagnostic, also inspect the raw patched-margin advantage:

\[
D_{\mathrm{RECOVERY}}
-
D_{\mathrm{AIE}}.
\]

This does not replace the primary normalized endpoint.

In [35]:
raw_primary = phase7_results[
    phase7_results[
        "noise_scale"
    ] == PRIMARY_SCALE
]

raw_by_case = (
    raw_primary
    .groupby(
        "case_id"
    )[
        "margin_advantage_RECOVERY"
    ]
    .mean()
)

raw_mean, raw_lo, raw_hi = (
    bootstrap_mean_ci(
        raw_by_case,
        seed=24680,
    )
)

print(
    "raw-margin cases:",
    len(raw_by_case),
)

print(
    "mean raw margin advantage:",
    raw_mean,
)

print(
    "raw margin 95% CI:",
    (
        raw_lo,
        raw_hi,
    ),
)

raw-margin cases: 128
mean raw margin advantage: 0.025775909423828125
raw margin 95% CI: (0.013951766490936279, 0.037832482655843096)


# 17. Save compact Phase-7 bridge summary

This file is intended to be the handoff from Phase 7 to Phase 8.

It records:

- frozen manifests and hashes;
- selected AIE and recovery-optimal pairs;
- descriptive synergy-optimal pair;
- primary holdout result;
- noise-scale robustness.

In [36]:
SUMMARY_PATH = (
    "phase7_bridge_summary_clean.json"
)

bridge_summary = {
    "phase7_version":
        PHASE7_VERSION,

    "model":
        MODEL_ID,

    "run_id":
        RUN_ID,

    "discovery_manifest":
        DISCOVERY_MANIFEST_PATH,

    "discovery_sha256":
        discovery_hash,

    "holdout_manifest":
        HOLDOUT_MANIFEST_PATH,

    "holdout_sha256":
        HOLDOUT_HASH,

    "AIE_pair":
        AIE_PAIR,

    "recovery_optimal_pair":
        RECOVERY_PAIR,

    "synergy_optimal_pair_descriptive":
        SYNERGY_PAIR,

    "primary_n_cases":
        int(
            len(primary_case)
        ),

    "primary_mean_recovery_AIE":
        float(
            primary_case[
                "recovery_AIE"
            ].mean()
        ),

    "primary_mean_recovery_RECOVERY":
        float(
            primary_case[
                "recovery_RECOVERY"
            ].mean()
        ),

    "primary_mean_delta":
        float(delta_mean),

    "primary_ci95":
        [
            float(lo),
            float(hi),
        ],

    "statistical_success":
        bool(lo > 0),

    "practical_threshold":
        float(
            PRACTICAL_THRESHOLD
        ),

    "practical_success":
        bool(
            delta_mean
            >= PRACTICAL_THRESHOLD
        ),

    "primary_win_rate":
        float(
            (
                primary_case[
                    "delta"
                ] > 0
            ).mean()
        ),

    "scale_summary":
        scale_summary.to_dict(
            orient="records"
        ),
}

with open(
    SUMMARY_PATH,
    "w",
) as f:
    json.dump(
        bridge_summary,
        f,
        indent=2,
    )

print(
    "Saved:",
    SUMMARY_PATH,
)

Saved: phase7_bridge_summary_clean.json


# Interpretation

Phase 7 should answer only the questions needed before Phase 8:

1. **Do the strongest first-order sites replicate on another Transformer?**
2. **Does a measured high-recovery pair improve over the AIE pair at equal budget?**
3. **Is that advantage robust to held-out subjects and multiple corruption strengths?**
4. **Is the high-recovery pair synergistic or redundant?**

The last distinction matters:

\[
J>0
\]

suggests complementary/super-additive causal interaction, while

\[
J<0
\]

with high pair recovery suggests overlapping or sequentially propagated information.

That makes Phase 7 a natural bridge:

\[
\boxed{
\text{Phase 6: interaction-aware causal recovery}
\rightarrow
\text{Phase 7: cross-model replication + held-out validation}
\rightarrow
\text{Phase 8: higher-order/context interactions}
}
\]

No additional replacement or post-hoc pair-selection cells should be inserted after the frozen selection is written.